##### Webscraping Program Curriculums
> Two Methods:
- **beautifulsoup (Python)**
- **selenium (JavaScript)**

In [ ]:
# Beautifulsoup import.
from bs4 import BeautifulSoup
from urllib.request import urlopen
from requests_html import HTMLSession

from urllib.error import HTTPError
import csv

from attr import attrs


In [ ]:
# Generate the bs object using requests * Correct for the JavaScript in rendering. 
# Attempt 2 due to JavaScript rendering.
import requests

url = "https://catalog.ncsu.edu/undergraduate/agriculture-life-sciences/food-bioprocessing-nutrition-science/nutrition-sciences-bs-applied-nutrition-concentration/#semestersequencetext"

response = requests.get(url)
response.raise_for_status()

bs = BeautifulSoup(response.text, "html.parser")

In [ ]:
# Creating the bs object. 
# try:
    # html = urlopen('https://catalog.ncsu.edu/undergraduate/agriculture-life-sciences/food-bioprocessing-nutrition-science/nutrition-sciences-bs-applied-nutrition-concentration/#semestersequencetext')
# except HTTPError as e:
    # print(e)
# except URLError as e:
    # print('The server could not be found!')
# else:
    # print('It Worked!')
# consider swapping the 'html.parser' for 'lxml' if you have it installed. It is faster.
# bs = BeautifulSoup(html, 'html.parser')

#### **Why am I getting info from the wrong?**

* * the code is not fetching the anchor (#semestersequencetext) because URL fragments are never sent to the server. *SO:* the server returns the base page, not the section you expected. 
* * when you make the URL request, everything but the # is a fragment identifier.  
* * The browser use fragments client-side only to scroll to the section of the page. 
* * Fragments are **NOT** sent in HTTP requests.  S0, the server will only receive the URL without the *#*.
* * **THEREFORE** *urlopen()* cannot fetch that subsection - it fetches the **whole page**. 
> * * Why the HTML looks different from what you see in the browser?
>> NCSU's catalog site is built with JavaScript that loads content dynamically:
>> When you fetch the page with Python:
>> * You get the **raw HTML**, befor JavaScript runs. 
>> * The "Semester Sequence" section is inserted by JavaScript ater page load. 
>> * urlopen() does not execute JavaScript. 
>> * So your **bs object** contains only the static HTML. 

> ##### TO VERIFY:
>> * * Open the page in the browser -> right-click -> "View Page Source". 
>> notice that the *Semester Sequence* is **not** contained in the raw HTML.
>> *It appears only in the rendered DOM, after JavaScript runs.*

##### Alternate Method to *CORRECTLY SCRAPE*

> * USE *requests_html* 

> *  Use Selenium:
>>from selenium import webdriver
>>driver = webdriver.Chrome()
>>driver.get('https://catalog.ncsu.edu/.../#semestersequencetext')
>>html = driver.page_source

> * Find API endpoint the site uses 
> *NCSU's catalog often loads sections from JSON endpoints.  You can inspect the Network tab in DevTools to find the real datasource*




In [65]:
if False:
    rows = bs.tbody.find_all("tr", class_=["even", "odd"])

    for row in rows:
        for cell in row.find_all("td"):
            match cell["class_"]:
                case "titlecol":
                    if cell:
                        cell1 = cell.get_text(strip=True)
                    else:
                        cell1 = "none"
                case "coursecol":
                    if cell:
                        cell2 = cell.get_text(strip=True)
                    else:
                        cell2 = "none"
                case "creditcol":
                    if cell:
                        cell3 = cell.get_text(strip=True)
                    else:
                        cell3 = "none"
                case _:
                    continue
        print(f'{cell1}, {cell2}, {cell3}')
print('ignore Attempt1')
            


ignore Attempt1


In [66]:
# Match attempt 2
if False:
    print(bs.tbody.tr.td.text)
    # print(bs.tbody.tr.get("class", ["even", "odd"])[0])
    print("\n")
    rows = bs.tbody.find_all("tr", class_=["even", "odd"])
    for row in rows:
        # print(row)
        i = 0 
        title = "error1"
        course = "error2"
        credits = "error3"
        for cell in row.find_all("td"): # This right here is creating a list.  
            # print(cell)
            match cell.get("class", []):
                case c if "titlecol" in c:
                    i = 1
                    title = cell.get_text(strip=True)
                    if title == "":
                        title = "Blank Line"
                case b if cell == row.find_all("td")[1]:
                # case c if "coursecol" in c:
                    i = 2
                    course = cell.get_text(strip=True)
                    if course == "":
                        course = "Blank Line"
                case c if "hourscol" in c:
                    i = 3
                    try:
                        credits = cell.get_text(strip=True)
                        if credits == "":
                            credits = "0"
                    except:
                        credits = "_"
                    
                case _:
                    match i: 
                        case 1:
                            course = "None"
                        case 2:
                            credits = "None"
                        case 3:
                            title = "None"
                        case 0:
                            title = bs.tbody.tr.td.text
                            course = ""
                            credits = ""
        print(f'{title}, {course}, {credits}')
print('Ignore Attempt2')

Ignore Attempt2


In [67]:
# Match attempt 3
# print(bs.tbody.tr.get("class", ["even", "odd"])[0])
if False:
    print("\n")
    rows = bs.tbody.find_all("tr", class_=["even", "odd"])
    for row in rows:
        # print(row)
        i = 0 
        title = "error1"
        course = "error2"
        credits = "error3"
        # if len(row.find_all("td")) < 3:
                # row.append("<td></td>")
        # The problem here is that sometimes the list isn't long enough. 
        for index, cell in enumerate(row.find_all("td")): # If I created a cells list, I could do "range(len(cells))"
            match index:
                case 0:
                    i = 1
                    title = cell.get_text(strip=True)
                    if title == "":
                        title = "Blank Line"
                case 1:
                    i = 2
                    course = cell.get_text(strip=True)
                    if course == "" and title == "Blank Line":
                        course = "Blank Line"
                    elif course == "":
                        course = "_"
                case 2:
                    i = 3
                    credits = cell.get_text(strip=True)
                    if credits == "" and title == "Blank Line":
                        credits = "Blank Line"
                    elif credits == "" and course == "_":
                        credits = "_"
                    else:
                        credits = "0"
                case _:
                    match i: 
                        case 1:
                            course = "ErrorA"
                        case 2:
                            credits = "ErrorB"
                        case 3:
                            title = "ErrorC"
                        case 0:
                            title = bs.tbody.tr.td.text
                            course = ""
                            credits = "_"
        print(f'[{title}], [{course}], [{credits}]')
print(f'Ignore Attempt3')

Ignore Attempt3


In [ ]:
# Match attempt 4
# I got this one to work good enough.
# Let me see how many pages it'll work for. 

rows = bs.tbody.find_all("tr", class_=["even", "odd"])
for row in rows:
    # print(row)
    i = 0 
    title = "error1"
    course = "error2"
    # credits = "error3"

    for index, cell in enumerate(row.find_all("td")): # I could do "range(len(cells))"
        match index:
            case 0:
                i = 1
                title = cell.get_text(strip=True)
                if title == "":
                    title = "Blank Line"
            case 1:
                i = 2
                course = cell.get_text(strip=True)
                # credits = "0" # Here is a working fix.
                if course == "" and title == "Blank Line":
                    course = "Blank Line"
                elif course == "":
                    course = "_"
                    credits = "_"
            case 2:
                i = 3
                credits = cell.get_text(strip=True)
                # if credits == "" and title == "Blank Line":
                    # credits = "Blank Line"
                # elif credits == "" and course == "_":
                    # credits = "_"
                if credits == "":
                    credits = "0"
            case _:
                match i: 
                    case 1:
                        course = "ErrorA"
                    case 2:
                        credits = "ErrorB"
                    case 3:
                        title = "ErrorC"
                    case 0: # Because the 0 case is clearly triggered.  Values should all be loaded. so credits should be over written.  
                        title = bs.tbody.tr.td.text
                        credits = "_"
                        course = "_"
                        
    print(f'[{title}], [{course}], [{credits}]')


[Exploring the Life Sciences], [_], [_]
[LSC 103], [Exploring Opportunities in the Life Sciences], [1]
[orALS 303], [Transfer Success in Agriculture and Life Sciences], [1]
[LSC 101], [Critical and Creative Thinking in the Life Sciences1, 2], [2]
[Communication], [3], [2]
[COM 110], [Public Speaking], [0]
[COM 112], [Interpersonal Communication], [0]
[ENG 333], [Communication for Science and Research], [0]
[Mathematics & Sciences], [_], [_]
[BIO 181], [Introductory Biology: Ecology, Evolution, and Biodiversity1], [4]
[BIO 183], [Introductory Biology: Cellular and Molecular Biology1], [4]
[BIO 240], [Principles of Human Anatomy & Physiology (A): Nervous, Skeletal, Muscular, & Digestive Systems], [3]
[CH 101&CH 102], [Chemistry - A Molecular Scienceand General Chemistry Laboratory1], [4]
[CH 201&CH 202], [Chemistry - A Quantitative Scienceand Quantitative Chemistry Laboratory], [4]
[CH 220&CH 222], [Introductory Organic Chemistryand Organic Chemistry I Lab], [4]
[orCH 221], [Organic Chem

# Print the file to PDF and use a PDF -> CSV reader -> xml. (fastest method)

##### Here I'm going to attempt to add my code to this CSV Section

In [ ]:
# Creating a CSV File of the Table. 
csvFile = open('nutrition_science.csv', 'wt+')
writer = csv.writer(csvFile)
try: 
    for row in rows:
        csvRow = []
        for cell in row.findAll('td', class_=["codecol","titlecol","hourscol"]):
            csvRow.append(cell.get_text())
        writer.writerow(csvRow)
finally:
    csvFile.close()

In [72]:
# This part should give me a CSV file called "nutrition_Science.csv"
csvFile = open('nutrition_Science.csv', 'wt+')
writer = csv.writer(csvFile)
rows = bs.tbody.find_all("tr", class_=["even", "odd"])

for row in rows:
    csvRow = []

    i = 0 
    title = "error1"
    course = "error2"
    # credits = "error3"

    for index, cell in enumerate(row.find_all("td")): # I could do "range(len(cells))"
        match index:
            case 0:
                i = 1
                title = cell.get_text(strip=True)
                if title == "":
                    title = "Blank Line"
            case 1:
                i = 2
                course = cell.get_text(strip=True)
                # credits = "0" # Here is a working fix.
                if course == "" and title == "Blank Line":
                    course = "Blank Line"
                elif course == "":
                    course = "_"
                    credits = "_"
            case 2:
                i = 3
                credits = cell.get_text(strip=True)
                # if credits == "" and title == "Blank Line":
                    # credits = "Blank Line"
                # elif credits == "" and course == "_":
                    # credits = "_"
                if credits == "":
                    credits = "0"
            case _:
                match i: 
                    case 1:
                        course = "ErrorA"
                    case 2:
                        credits = "ErrorB"
                    case 3:
                        title = "ErrorC"
                    case 0: # Because the 0 case is clearly triggered.  Values should all be loaded. so credits should be over written.  
                        title = bs.tbody.tr.td.text
                        credits = "_"
                        course = "_"
                        
    csvRow.append([title, course, credits])
    writer.writerow(csvRow)

csvFile.close()

##### In the Next Section I will use JavaScript
> pip install webdriver-manager

In [ ]:
# Working with Selenium JavaScript
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Create a new instance of the Chrome driver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get('https://catalog.ncsu.edu/undergraduate/agriculture-life-sciences/food-bioprocessing-nutrition-science/nutrition-sciences-bs-applied-nutrition-concentration/#semestersequencetext')
time.sleap(2)
driver.close

# Go to the desired webpage
driver.get('https://catalog.ncsu.edu/undergraduate/agriculture-life-sciences/
food-bioprocessing-nutrition-science/nutrition-sciences-bs-applied-nutrition-concentration/#semestersequencetext')

# Extract the page source after JavaScript has rendered
html = driver.page_source



In [ ]:
# This is Revised code from earlier for scraping the "correct" target page:
from selenium import webdriver
driver = webdriver.Chrome()
driver.get('https://catalog.ncsu.edu/.../#semestersequencetext')
html = driver.page_source
# I believe the problem with that the # was not read as the anchor for the bs python.  So the page selected is prioritized incorrectly. 


##### In the next section I will use a pandas web reader.

In [ ]:
# Alternate method using pandas.
# Looks like there are a couple more steps with this one. 
import pandas as pd

url = "https://catalog.ncsu.edu/undergraduate/agriculture-life-sciences/food-bioprocessing-nutrition-science/nutrition-sciences-bs-applied-nutrition-concentration/#semestersequencetext"

tables = pd.read_html(url)
len(tables)


tables[0].to_csv("nutrition_sciences(2).csv", index=False)
